# R4 · P1 y P2 (cada respuesta: número, gráfico y 2 o 3 líneas)

**Responsable:** Leo (Rol D)  
**Revisor:** Dan (Rol B)

## Qué queda resuelto

- [x] P1 · Chile frente a Sudamérica: casos y fallecidos por 100.000 (media móvil 7 días); olas, inicio, fin y la peor
- [x] P2 · Regiones de Chile por ola desde jun-2020; posición de Biobío; mapa regional

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import duckdb
import pandas as pd
from IPython.display import Image, display

DW_PATH = str(RAIZ / 'data' / 'dw_covid.duckdb')
con = duckdb.connect(DW_PATH, read_only=True)

## P1 · Chile frente a Sudamérica: casos y fallecidos por 100.000; olas en Chile

In [ ]:
sudamerica = "'Argentina','Bolivia','Brazil','Chile','Colombia','Ecuador','Paraguay','Peru','Uruguay','Venezuela'"
p1 = con.execute(f'''
    WITH pob AS (
        SELECT country_region,
               COALESCE(MAX(CASE WHEN nivel_granularidad='pais' THEN population END),
                        SUM(CASE WHEN nivel_granularidad='region_estado' THEN population END), SUM(population)) AS population
        FROM dim_territorio WHERE country_region IN ({sudamerica}) AND NOT es_entidad_especial GROUP BY 1
    ),
    ult AS (
        SELECT t.country_region, SUM(f.confirmed) AS confirmed, SUM(f.deaths) AS deaths
        FROM fact_covid f JOIN dim_territorio t ON f.territorio_key = t.territorio_key
        WHERE f.fecha_key = 20230309 AND t.country_region IN ({sudamerica}) GROUP BY 1
    )
    SELECT u.country_region, ROUND(u.confirmed * 100000.0 / p.population, 1) AS casos_100k,
           ROUND(u.deaths * 100000.0 / p.population, 1) AS fallecidos_100k
    FROM ult u JOIN pob p ON u.country_region = p.country_region ORDER BY fallecidos_100k DESC
''').df()
display(p1)
display(Image(filename=str(RAIZ / 'docs' / 'figuras' / 'p1_sudamerica.png')))
# Chile cerró 2.° en casos/100k (27.161,7) y 2.° en fallecidos/100k (336,2) en Sudamérica, con 4 olas principales
# (may-jul 2020, feb-jun 2021, ene-mar 2022 ómicron —la mayor en casos diarios— y may-jul 2022).

## P2 · Regiones de Chile por ola desde jun-2020; posición de Biobío; mapa regional

In [ ]:
p2 = con.execute('''
    SELECT t.province_state AS region, t.population, SUM(f.confirmed) AS confirmed, SUM(f.deaths) AS deaths,
           ROUND(SUM(f.confirmed) * 100000.0 / NULLIF(t.population, 0), 1) AS casos_100k,
           ROUND(SUM(f.deaths) * 100000.0 / NULLIF(t.population, 0), 1) AS fallecidos_100k
    FROM fact_covid f JOIN dim_territorio t ON f.territorio_key = t.territorio_key
    WHERE t.country_region = 'Chile' AND t.province_state IS NOT NULL AND t.province_state <> 'Unknown' AND f.fecha_key = 20230309
    GROUP BY 1, 2 ORDER BY casos_100k DESC
''').df()
con.close()
display(p2)
display(Image(filename=str(RAIZ / 'docs' / 'figuras' / 'p2_regiones_chile.png')))
# Magallanes lidera en casos (40.624,4/100k) y fallecidos (464,8/100k). Biobío ocupa el puesto #9 en casos/100k (32.806,3) y #10 en fallecidos/100k (309,9).